In [ ]:
import os
import json
import random
import pandas as pd
from pathlib import Path

def process_site(root: Path):
    cn, adhd = [], []
    tsv_file = list(root.glob("*tsv"))[0]
    
    df = None
    for enc in ['utf-8', 'latin1', 'cp1252', 'gbk']:
        try:
            df = pd.read_csv(tsv_file, sep='\t', encoding=enc)
            print(f"Successfully read {tsv_file.name} with {enc}")
            break
        except UnicodeDecodeError:
            continue
    # print(len(df['participant_id'].unique()))
    # print(len(os.listdir(str(root)))-1)
    df['participant_id'] = df['participant_id'].astype(str)
    count = 0
    for participant_dir in root.glob("*"):
        if not participant_dir.is_dir():
            continue
        participant_id = str(int(participant_dir.name.split("-")[1]))
        dx_serie = df.loc[df['participant_id'] == participant_id, 'dx'].iloc[0]

        nii_dir = participant_dir / "ses-1" / "anat"
        nii_files = list(nii_dir.glob("*.nii*"))
        if len(nii_files) == 0:
            count += 1
            continue
        
        nii_file = nii_files[-1]
        
        sample = {
            "images": [str(nii_file)],
            "label": 0 if dx_serie == "Typically Developing Children" else 1
        }
        if sample["label"] == 0:
            cn.append(sample)
        else:
            adhd.append(sample)
    print(f"count: {count}")
    return cn, adhd

def process_adhd(data_root, output_path, ratio=0.7):
    cn_all, adhd_all = [], []
    data_root = Path(data_root)
    for site in data_root.glob("*"):
        if not site.is_dir():
            continue
        site_cn, site_adhd = process_site(site)
        cn_all.extend(site_cn)   
        adhd_all.extend(site_adhd)
    
    n_cn, n_adhd = len(cn_all), len(adhd_all)
    print(n_cn, n_adhd, n_cn+n_adhd)
    n_balanced = min(n_cn, n_adhd)
    balanced_cn = random.sample(cn_all, n_balanced)
    balanced_adhd = random.sample(adhd_all, n_balanced)
    
    n_train = int(ratio * n_balanced)
    train_samples = balanced_cn[:n_train] + balanced_adhd[:n_train]
    val_samples = balanced_cn[n_train:] + balanced_adhd[n_train:]
    
    random.shuffle(train_samples)
    random.shuffle(val_samples)
    print(len(train_samples))
    print(len(val_samples))
    data = {"train": train_samples, "val": val_samples}
    output_path = Path(output_path)
    output_path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")


data_root = r"/media/bmp10/da6a1340-5b8d-4bc2-9196-3a3bf678f27f/students/undergraduate/2022/WHF/datasets/adhd"
output_path = r"/media/bmp10/da6a1340-5b8d-4bc2-9196-3a3bf678f27f/students/undergraduate/2022/WHF/discrep-mamaba/jsons/adhd.json"
process_adhd(data_root, output_path) 

Successfully read participants.tsv with latin1
count: 5
Successfully read participants.tsv with utf-8
count: 1
Successfully read participants.tsv with utf-8
count: 1
Successfully read participants.tsv with utf-8
count: 10
Successfully read participants.tsv with utf-8
count: 1
Successfully read participants.tsv with utf-8
count: 0
Successfully read participants.tsv with utf-8
count: 0
Successfully read participants.tsv with utf-8
count: 1
Successfully read participants.tsv with utf-8
count: 3
559 354 913
494
214
